# CPU verification of the backhoe loader and excavator pipeline

This short notebook is the fallback when a fresh Google Colab session cannot obtain a GPU. It downloads and verifies the **same frozen Roboflow version 3 dataset**, trains YOLO26n for **five CPU epochs** as a pipeline check, and then **loads the separately released `best.pt` from the original 30-epoch T4 run** for held-out test evaluation and example predictions.

The five-epoch checkpoint is **not** the model behind the published scores or PDF figures. The [full training notebook](01_BHL_EXC_Object_Detection_Training.ipynb) records how to regenerate the 30-epoch checkpoint when a GPU is available. This notebook needs no account, API key, secret, or local install.

## 1. Install and record the environment

Colab provides Python and PyTorch. This cell pins Ultralytics to the version used in the reported run and records the available device. The verification below explicitly uses the CPU.

In [ ]:
%pip install -q ultralytics==8.4.163 pyyaml

import platform
import time
import torch
import ultralytics
from datetime import datetime, timezone

proof_started = time.perf_counter()
print("Started (UTC):", datetime.now(timezone.utc).isoformat(timespec="seconds"))
print("Ultralytics:", ultralytics.__version__)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Verification device: CPU")

## 2. Download the fixed dataset and check its fingerprint

The public GitHub Release holds the 131-image YOLOv11-format export. A SHA256 check proves that the downloaded ZIP is byte-for-byte the frozen file used for the reported experiment. The export format and the **YOLO26n** model version describe different things.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import urllib.request
import yaml

DATA_URL = "https://github.com/Asiya0555/Backhoe-Excavator-Site-Detection/releases/download/v1.0.0/backhoe-excavator-v3-yolov11.zip"
EXPECTED_SHA256 = "8e3645e885a7f4b3ec90c25cdf39b680a474448659397f123ac13a7d28ccab8a"
ZIP_PATH = Path("/content/backhoe-excavator-v3-yolov11.zip")
ROOT = Path("/content/dataset")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

if ZIP_PATH.exists() and sha256_file(ZIP_PATH) != EXPECTED_SHA256:
    ZIP_PATH.unlink()
if not ZIP_PATH.exists():
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

actual_sha256 = sha256_file(ZIP_PATH)
if actual_sha256 != EXPECTED_SHA256:
    ZIP_PATH.unlink(missing_ok=True)
    raise AssertionError(f"Dataset checksum mismatch: {actual_sha256}")
print("Dataset SHA256 verified:", actual_sha256)

ROOT.mkdir(parents=True, exist_ok=True)
with ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None, "ZIP is damaged"
    archive.extractall(ROOT)

yaml_files = [p for p in ROOT.rglob("data.yaml")
              if (p.parent / "train" / "images").is_dir()]
assert len(yaml_files) == 1, f"Expected one dataset configuration: {yaml_files}"
cfg_path = yaml_files[0]
dataset_root = cfg_path.parent
cfg = yaml.safe_load(cfg_path.read_text())
cfg.update(path=str(dataset_root), train="train/images", val="valid/images", test="test/images")
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
assert cfg["names"] == ["Backhoe Loader", "excavator"]
for split, expected in (("train", 105), ("valid", 13), ("test", 13)):
    n = len(list((dataset_root / split / "images").glob("*.jpg")))
    assert n == expected, (split, n)
    print(f"{split}: {n} images")
print("Data configuration:", cfg_path)

## 3. Run a five-epoch CPU training check

This step confirms that the frozen labels, model, and training command work together in a fresh session. Its validation curve and temporary weights belong **only to this five-epoch check**. The actual reported model was trained for 30 epochs on a T4 with batch size 16, image size 640, and seed 42.

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display

verification_model = YOLO("yolo26n.pt")
train_started = time.perf_counter()
verification_metrics = verification_model.train(
    data=str(cfg_path),
    epochs=5,
    batch=8,
    imgsz=640,
    device="cpu",
    seed=42,
    workers=2,
    project="/content/runs",
    name="cpu_verification_v3_yolo26n",
    plots=True,
)
verification_minutes = (time.perf_counter() - train_started) / 60
verification_dir = Path(verification_model.trainer.save_dir)
verification_weights = Path(verification_model.trainer.best)
assert verification_weights.is_file(), "Five-epoch verification weights missing"
print(f"Five-epoch CPU training took {verification_minutes:.1f} minutes")
print("Verification weights (not the published checkpoint):", verification_weights)
print("Verification results:", verification_dir)
if (verification_dir / "results.png").is_file():
    display(Image(filename=str(verification_dir / "results.png"), width=750))

## 4. Load the published 30-epoch checkpoint and evaluate the test split

The Release asset `best.pt` was selected using validation results during the original T4 run. It is downloaded separately so this CPU check does not silently substitute its five-epoch weights. Test precision, recall, mAP50, and mAP50-95 below are computed from **the released checkpoint** on the 13 held-out photos. Small numerical differences from the published run are possible across environments.

In [ ]:
MODEL_URL = "https://github.com/Asiya0555/Backhoe-Excavator-Site-Detection/releases/download/v1.0.0/best.pt"
RELEASED_WEIGHTS = Path("/content/released_30_epoch_best.pt")
if not RELEASED_WEIGHTS.is_file():
    urllib.request.urlretrieve(MODEL_URL, RELEASED_WEIGHTS)
assert RELEASED_WEIGHTS.stat().st_size > 100_000, "Released checkpoint looks incomplete"
assert RELEASED_WEIGHTS.resolve() != verification_weights.resolve()

reported_model = YOLO(str(RELEASED_WEIGHTS))
test_metrics = reported_model.val(
    data=str(cfg_path), split="test", imgsz=640, batch=8, device="cpu",
    plots=True, project="/content/runs", name="released_model_test_cpu", exist_ok=True,
)
print("Checkpoint evaluated:", RELEASED_WEIGHTS)
print(f"Precision: {test_metrics.box.mp:.3f}")
print(f"Recall: {test_metrics.box.mr:.3f}")
print(f"mAP50: {test_metrics.box.map50:.3f}")
print(f"mAP50-95: {test_metrics.box.map:.3f}")
print("Test plots:", test_metrics.save_dir)
matrix = Path(test_metrics.save_dir) / "confusion_matrix.png"
if matrix.is_file():
    display(Image(filename=str(matrix), width=650))

## 5. Check three held-out photographs

These fixed test photos were excluded from training: one backhoe loader, one negative tractor, and one excavator. The negative may receive a false target box at confidence 0.25. That is a known error of the published model, not a code failure.

In [ ]:
import matplotlib.pyplot as plt

cases = [
    ("Backhoe loader, rear view", "37874580040398070775d54550f63135"),
    ("Negative front-loader tractor", "8af5b8abe123249eeda2bcb818d58f43"),
    ("Excavator, site view", "8887f5372e6aa57f98099b2a78dcf2d1"),
]
test_image_dir = dataset_root / "test" / "images"
photos = []
for title, file_id in cases:
    matches = list(test_image_dir.glob(f"*.rf.{file_id}.jpg"))
    assert len(matches) == 1, f"Missing fixed test photo: {title}"
    photos.append(matches[0])

predictions = reported_model.predict(
    source=[str(p) for p in photos], imgsz=640, conf=0.25,
    device="cpu", verbose=False,
)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (title, _), result in zip(axes, cases, predictions):
    ax.imshow(result.plot()[:, :, ::-1])
    ax.set_title(title)
    ax.axis("off")
    labels = [f"{result.names[int(cls)]} {score:.2f}"
              for cls, score in zip(result.boxes.cls.cpu().tolist(),
                                    result.boxes.conf.cpu().tolist())]
    print(title, "|", ", ".join(labels) or "none")
plt.tight_layout()
plt.show()
print("Completed (UTC):", datetime.now(timezone.utc).isoformat(timespec="seconds"))
print(f"Total notebook time after installation: {(time.perf_counter()-proof_started)/60:.1f} minutes")

## What this proves

The five-epoch training check verifies that the public, SHA256-verified data can train in a fresh CPU Colab runtime. The held-out evaluation and three predictions use the **released 30-epoch checkpoint**. The [full training notebook](01_BHL_EXC_Object_Detection_Training.ipynb) contains the original parameters and complete results; the [repository evidence](../results/README.md) includes its 30-epoch `results.png`, annotation samples, validation grid, five-photo demonstration, and error analysis. This short check does not claim that five epochs reproduce the 30-epoch model.